# OR–odorant functional landscape — v3

Four activation matrices, each clustered (Jaccard + PAM, k=6) with hypergeometric + BH enrichment:
- **baseline** — predictions, p ≥ 0.50
- **calibrated** — predictions, p > 0.83
- **hybrid_0.5** — M2OR ground truth (exact-sequence matched) over the p ≥ 0.50 fill
- **hybrid_0.83** — M2OR ground truth over the p > 0.83 fill

Per matrix: `heatmap_{name}` (rows in co-tuning order), `heatmap_{name}_tree_order` (rows in phylogenetic order), `spider_{name}` and `enrichment_{name}` (two views of the same enrichment). Plus `chemspace` (Class I vs II ligand descriptors). All figures exported as SVG.

In [1]:
import numpy as np, pandas as pd, matplotlib, re
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import hypergeom, mannwhitneyu
from sklearn.metrics import silhouette_score
from collections import Counter
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, Lipinski
np.random.seed(0)
plt.rcParams.update({'font.family':'Liberation Sans','pdf.fonttype':42,'ps.fonttype':42,
    'axes.linewidth':0.8,'axes.edgecolor':'#c9c9c4','text.color':'#8a8a86','axes.labelcolor':'#c9c9c4',
    'xtick.color':'#8a8a86','ytick.color':'#8a8a86','savefig.dpi':300})

PROB  = "/home/izirdeli/Documents.cglab/olfactory-receptors/Downstream_Analysis/predictions/aggregated/Reference_Tree_Predictions/reference_median_probability_wide.csv"
PAIRS = "/home/izirdeli/Documents.cglab/olfactory-receptors/Model/Data_Preparation/processed/m2or_pairs_model.csv"                     # smiles_id <-> SMILES, experimental Responsive, Sequence
ODORS = "/home/izirdeli/Documents.cglab/DownStreamAnalysis/01.Data/metadata/inchi_smiles_odors.csv"                  # SMILES -> Odors
FASTA = "/home/izirdeli/Documents.cglab/olfactory-receptors/Phylogenetic_Analysis/data/ReferenceTree/PF13853.9606_7955_7740_7764_75743_137246.fa"  # reference sequences, headers = protein_id
CLASSES="/home/izirdeli/Documents.cglab/olfactory-receptors/Phylogenetic_Analysis/data/HumanTree/human433_OR_classes.csv" 
TREE="/home/izirdeli/Documents.cglab/olfactory-receptors/Phylogenetic_Analysis/data/HumanTree/human433_MFP_reorder.tree"
CLASS_I_COL='#7f1734'; CLASS_II_COL='#557c99'
OUTDIR = 'Figures'
K = 6
CALIB_THR = 0.83

## 1 · Data: probability matrix, human ORs

In [2]:
import os; os.makedirs(OUTDIR, exist_ok=True)
prob=pd.read_csv(PROB,index_col=0)
hum_pids=[i for i in prob.index if str(i).startswith('9606')]
P=prob.loc[hum_pids]; lig_cols=list(P.columns); Pv=P.values
print('human probability matrix:',Pv.shape)

human probability matrix: (433, 754)


## 2 · Experimental layer — exact amino-acid sequence match (23,782 pairs)

In [3]:
pid2seq={}; pid=None; buf=[]
for line in open(FASTA):
    line=line.rstrip()
    if line.startswith('>'):
        if pid: pid2seq[pid]=''.join(buf).upper()
        pid=line[1:].split()[0]; buf=[]
    else: buf.append(line)
if pid: pid2seq[pid]=''.join(buf).upper()
seq2pid={}
for p in hum_pids: seq2pid.setdefault(pid2seq[p],[]).append(p)
m=pd.read_csv(PAIRS,usecols=['Species','UniProt ID','Sequence','Responsive','smiles_id','mutation'])
hs=m[m['Species'].str.lower().str.contains('homo')].copy()
hs['Sequence']=hs['Sequence'].str.upper().str.strip()
hs['pid']=hs['Sequence'].map(lambda s: seq2pid.get(s,[None])[0])
matched=hs[hs['pid'].notna() & hs['smiles_id'].isin(set(lig_cols))]
exp=matched.groupby(['pid','smiles_id'])['Responsive'].max().reset_index()
print('overlapping human pairs:',len(exp),'(ground truth 23,782)')

overlapping human pairs: 23782 (ground truth 23,782)


## 3 · Odor-tag map

In [4]:
pairs=pd.read_csv(PAIRS,usecols=['SMILES','smiles_id'])
sml2smiles=pairs.drop_duplicates('smiles_id').set_index('smiles_id')['SMILES'].to_dict()
odo=pd.read_csv(ODORS)
smiles2tags={s:[t.strip().lower() for t in o.split(',') if t.strip()]
             for s,o in zip(odo['SMILES'],odo['Odors']) if isinstance(o,str) and o.strip()}
sml2tags_all={sid:smiles2tags[sml2smiles[sid]] for sid in lig_cols if sml2smiles.get(sid) in smiles2tags}
print('tagged odorants:',len(sml2tags_all),'/',len(lig_cols))

tagged odorants: 575 / 754


## 4 · Receptor classes and phylogenetic row order

In [5]:
cls=pd.read_csv(CLASSES)
pid2class={r.leaf_id:('I' if r.OR_class.split('_')[1]=='I' else 'II') for r in cls.itertuples()}
n_I=sum(v=='I' for v in pid2class.values()); n_II=sum(v=='II' for v in pid2class.values())
tree_order=re.findall(r'[\(,]([^(),:;]+):', open(TREE).read())   # ladderized tip order
assert set(tree_order)==set(hum_pids)
print('Class I:',n_I,'Class II:',n_II,'| tree tips:',len(tree_order))

Class I: 62 Class II: 371 | tree tips: 433


## 5 · Palette and helpers

In [6]:
theme_rules = [
    (('acidic','sour','cheesy','dairy'),                       'Acid / Dairy / Cheese',       '#543005'),
    (('chemical','potato','vanilla','roasted','phenolic','popcorn','spicy'),'Roasted / Phenolic / Spice','#ae7121'),
    (('plum','jasmine','floral','balsamic'),                   'Floral / Balsamic',           '#e7cf94'),
    (('lavender','violet','green','fresh','herbal','citrus'),  'Fresh / Green / Herbal',      '#98d7cd'),
    (('cognac','coconut','pineapple','aldehydic','fruity','waxy','buttery'),'Fruity / Fatty / Aldehydic','#98d7cd'),
    (('alcoholic','ethereal','medicinal','fermented','smoky','rum','solvent'),'Fermented / Solvent','#24877f'),
    (('musk','animal','grapefruit'),                           'Musk / Animal',               '#003c30')]
SHORT={'Acid / Dairy / Cheese':'Acid·Dairy','Roasted / Phenolic / Spice':'Roasted·Spice','Floral / Balsamic':'Floral',
       'Fresh / Green / Herbal':'Fresh·Green','Fruity / Fatty / Aldehydic':'Fruity·Fatty','Fermented / Solvent':'Fermented','Musk / Animal':'Musk·Animal'}

def save(fig,name):
    fig.savefig(f'{OUTDIR}/{name}.svg', bbox_inches='tight'); plt.close(fig)

def kmedoids(D,k,n_init=15,seed=0):
    rng=np.random.default_rng(seed); n=len(D); best=None
    for _ in range(n_init):
        med=rng.choice(n,k,replace=False)
        for _ in range(300):
            lab=D[:,med].argmin(1); nm=med.copy()
            for c in range(k):
                idx=np.where(lab==c)[0]
                if len(idx): nm[c]=idx[D[np.ix_(idx,idx)].sum(1).argmin()]
            if set(nm)==set(med): med=nm; break
            med=nm
        lab=D[:,med].argmin(1); cost=D[np.arange(n),med[lab]].sum()
        if best is None or cost<best[0]: best=(cost,lab.copy())
    return best[1]

def bh(p):
    p=np.asarray(p); m=len(p); o=np.argsort(p); r=p[o]
    q=np.minimum.accumulate((r*m/(np.arange(m)+1))[::-1])[::-1]; out=np.empty(m); out[o]=np.clip(q,0,1); return out

def cluster_and_enrich(X,K=6):
    cols=np.array(lig_cols); keep=X.sum(0)>0; lig,Xn=cols[keep],X[:,keep]
    Dj=np.nan_to_num(squareform(pdist(Xn.T,'jaccard'))); lab=kmedoids(Dj,K)
    s2t={s:sml2tags_all[s] for s in lig if s in sml2tags_all}
    tagged=np.array([s in s2t for s in lig]); tagN=Counter([t for s in lig[tagged] for t in s2t[s]]); N=int(tagged.sum())
    tags=[t for t,c in tagN.items() if c>=5]; rows=[]
    for c in range(K):
        mem=[s for s in lig[lab==c] if s in s2t]; n=len(mem); ct=Counter([t for s in mem for t in s2t[s]])
        for t in tags:
            kk=ct.get(t,0); Kt=tagN[t]; e=n*Kt/N; oe=kk/e if e>0 else 0
            p=hypergeom.sf(kk-1,N,Kt,n) if kk>0 else 1.0
            rows.append([c,n,t,kk,Kt,round(e,3),round(oe,3),p])
    enr=pd.DataFrame(rows,columns=['cluster','n','tag','obs','tag_total','exp','obs_exp','p'])
    enr['fdr']=bh(enr['p'].values); sig=enr[(enr.fdr<0.05)&(enr.obs_exp>1)]
    # bijective theme assignment: every cluster gets a DISTINCT colour
    score={}
    for c in range(K):
        s=sig[sig.cluster==c]
        for ti,(keys,nm,col) in enumerate(theme_rules):
            hit=s[s.tag.isin(keys)]
            if len(hit): score[(c,ti)]=float(hit.obs_exp.max())
    meta={c:None for c in range(K)}; used=set()
    for (c,ti),sc in sorted(score.items(),key=lambda x:-x[1]):
        keys,nm,col=theme_rules[ti]
        if meta[c] is None and col not in used:
            meta[c]=dict(name=nm,color=col,size=int((lab==c).sum())); used.add(col)
    palette=list(dict.fromkeys(col for _,_,col in theme_rules))
    for c in range(K):
        if meta[c] is None:
            col=next((x for x in palette if x not in used),'#8a8a8a')
            top=sig[sig.cluster==c].sort_values('obs_exp',ascending=False).tag.tolist()
            meta[c]=dict(name=top[0].capitalize() if top else f'cluster{c}',color=col,size=int((lab==c).sum())); used.add(col)
    order={col:i for i,col in enumerate(palette)}
    cl_order=sorted(range(K),key=lambda c: order.get(meta[c]['color'],99))
    return lig,Xn,lab,enr,sig,meta,cl_order

### Activation heatmap — class stripe (left), breadth (right); rows in co-tuning **or** phylogenetic order

In [7]:
def activation_heatmap(lig,Xn,lab,meta,cl_order,name,title,subtitle,roworder='cotuning',K=6):
    if roworder=='tree':
        ridx={p:i for i,p in enumerate(hum_pids)}; row_pos=np.array([ridx[p] for p in tree_order])
        ylab='433 human ORs (phylogenetic order)'; fname=f'heatmap_{name}_tree_order'
    else:
        row_pos=leaves_list(linkage(squareform(np.nan_to_num(squareform(pdist(Xn,'jaccard')))),'average'))
        ylab='433 human ORs (co-tuning order)'; fname=f'heatmap_{name}'
    col_index=[]; bounds=[]; pos=0
    for c in cl_order:
        idx=np.where(lab==c)[0]
        if len(idx)>2: idx=idx[leaves_list(linkage(np.nan_to_num(pdist(Xn[:,idx].T,'jaccard')),'average'))]
        col_index+=idx.tolist(); pos+=len(idx); bounds.append(pos)
    col_index=np.array(col_index); M=Xn[np.ix_(row_pos,col_index)]; starts=[0]+bounds[:-1]
    fig=plt.figure(figsize=(12.4,7.6))
    gs=fig.add_gridspec(2,3,width_ratios=[0.11,4,0.11],height_ratios=[0.14,4],hspace=0.028,wspace=0.02,left=0.055,right=0.94,top=0.83,bottom=0.135)
    ax=fig.add_subplot(gs[1,1])
    ax.imshow(M,aspect='auto',cmap=ListedColormap(['#F4F1EA','#2E4057']),interpolation='nearest',rasterized=True)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_edgecolor('#bbb'); sp.set_linewidth(0.6)
    for i in range(1,len(bounds)): ax.axvline(starts[i]-0.5,color='white',lw=1.8)
    ax.set_xlabel(f'{len(lig)} odorants — co-tuning cluster (Jaccard, PAM, k={K})',fontsize=10.5,labelpad=8,color='#8a8a86')
    ax.set_ylabel(ylab,fontsize=10.5,color='#8a8a86')
    axtop=fig.add_subplot(gs[0,1],sharex=ax)
    axtop.imshow(np.array([[matplotlib.colors.to_rgb(meta[lab[j]]['color']) for j in col_index]]),aspect='auto',interpolation='nearest',extent=[0,len(col_index),0,1])
    axtop.set_xlim(0,len(col_index)); axtop.set_ylim(0,1); axtop.set_xticks([]); axtop.set_yticks([])
    for sp in axtop.spines.values(): sp.set_visible(False)
    for i,c in enumerate(cl_order):
        axtop.text((starts[i]+bounds[i])/2,1.55,SHORT.get(meta[c]['name'],meta[c]['name'][:10]),ha='center',va='bottom',fontsize=9,color=meta[c]['color'],fontweight='bold',clip_on=False)
    axcls=fig.add_subplot(gs[1,0],sharey=ax)
    axcls.imshow(np.array([[matplotlib.colors.to_rgb(CLASS_I_COL if pid2class[hum_pids[j]]=='I' else CLASS_II_COL)] for j in row_pos]),aspect='auto',interpolation='nearest')
    axcls.set_xticks([]); axcls.set_yticks([])
    for sp in axcls.spines.values(): sp.set_visible(False)
    axcls.set_xlabel('class',fontsize=7.5,color='#8a8a86',labelpad=4)
    axbr=fig.add_subplot(gs[1,2],sharey=ax)
    axbr.imshow(Xn.sum(1)[row_pos].reshape(-1,1),aspect='auto',cmap='Greys',interpolation='nearest',rasterized=True)
    axbr.set_xticks([]); axbr.set_yticks([])
    for sp in axbr.spines.values(): sp.set_visible(False)
    axbr.set_xlabel('breadth',fontsize=7.5,color='#8a8a86',labelpad=4)
    ax.legend(handles=[Patch(fc=CLASS_I_COL,label='Class I'),Patch(fc=CLASS_II_COL,label='Class II'),Patch(fc='#2E4057',label='binding'),Patch(fc='#F4F1EA',ec='#bbb',label='no binding')],
              loc='upper left',bbox_to_anchor=(0,-0.05),ncol=4,frameon=False,fontsize=9,handlelength=1.1,columnspacing=1.4,labelcolor='#8a8a86')
    fig.suptitle(title,fontsize=12.5,y=0.965,x=0.5,fontweight='bold',color='#6b6b67')
    fig.text(0.5,0.905,subtitle,ha='center',fontsize=9.5,color='#a5a5a0')
    save(fig,fname)

### Spider (radar) enrichment

In [8]:
def spider(enr,sig,meta,cl_order,name,title):
    axis_tags=[]; sector=[]
    for c in cl_order:
        cnt=0
        for t in sig[sig.cluster==c].sort_values('obs_exp',ascending=False).tag.tolist():
            if t not in axis_tags: axis_tags.append(t); sector.append(c); cnt+=1
            if cnt>=6: break
    if len(axis_tags)<3: return
    oe=enr.pivot(index='cluster',columns='tag',values='obs_exp').fillna(0)
    ang=np.linspace(0,2*np.pi,len(axis_tags),endpoint=False); rmax=max(6,float(np.ceil(sig['obs_exp'].max())))
    fig=plt.figure(figsize=(9.6,9.6)); axp=fig.add_subplot(111,polar=True)
    axp.set_theta_offset(np.pi/2); axp.set_theta_direction(-1); axp.set_ylim(0,rmax+1)
    ticks=list(range(1,int(rmax)+1,max(1,int(rmax)//6))); axp.set_yticks(ticks)
    axp.set_yticklabels([f'{t}×' for t in ticks],fontsize=7.5,color='#a5a5a0')
    axp.set_xticks(ang); axp.set_xticklabels([]); axp.spines['polar'].set_color('#d5d5d0'); axp.grid(color='#e6e6e2',lw=0.7)
    axp.plot(np.linspace(0,2*np.pi,200),[1]*200,color='#b5b5b0',lw=1.0,ls=(0,(4,3)),zorder=2)
    ac=np.concatenate([ang,ang[:1]])
    for c in cl_order:
        v=np.array([oe.loc[c,t] if t in oe.columns else 0 for t in axis_tags]); v=np.concatenate([v,v[:1]])
        axp.plot(ac,v,color=meta[c]['color'],lw=2.4,zorder=5,solid_joinstyle='round'); axp.fill(ac,v,color=meta[c]['color'],alpha=0.13,zorder=3)
    for a,t,c in zip(ang,axis_tags,sector):
        deg=np.degrees(a); rot=90-deg; ha='left'
        if 90<deg<270: rot+=180; ha='right'
        axp.text(a,rmax+1.2,t,rotation=rot,rotation_mode='anchor',ha=ha,va='center',fontsize=8.2,color=meta[c]['color'],fontweight='bold')
    axp.legend(handles=[plt.Line2D([0],[0],color=meta[c]['color'],lw=3,label=f"{meta[c]['name']} (n={meta[c]['size']})") for c in cl_order],
               loc='upper center',bbox_to_anchor=(0.5,-0.055),ncol=2,frameon=False,fontsize=9.3,handlelength=1.6,columnspacing=2.2,labelspacing=0.6,labelcolor='#6b6b67')
    fig.text(0.5,0.965,title,ha='center',fontsize=13,fontweight='bold',color='#6b6b67')
    fig.text(0.5,0.935,'radius = observed / expected odor-tag frequency  ·  FDR < 0.05',ha='center',fontsize=9.5,color='#a5a5a0')
    save(fig,f'spider_{name}')

### Enrichment heatmap (dotted, log₂ fold)

In [9]:
def enrichment_heatmap(enr,sig,meta,cl_order,name,title):
    sig_tags=sig.tag.unique().tolist()
    owner={}; strength={}
    for t in sig_tags:
        sub=enr[(enr.tag==t)&(enr.cluster.isin(cl_order))]; best=sub.loc[sub.obs_exp.idxmax()]
        owner[t]=int(best.cluster); strength[t]=best.obs_exp
    tags=sorted(sig_tags,key=lambda t:(cl_order.index(owner[t]),-strength[t]))
    oe=enr.pivot(index='cluster',columns='tag',values='obs_exp'); fdr=enr.pivot(index='cluster',columns='tag',values='fdr')
    L=np.zeros((len(cl_order),len(tags)))
    for i,c in enumerate(cl_order):
        for j,t in enumerate(tags):
            v=oe.loc[c,t] if t in oe.columns else 0; L[i,j]=max(0.0,np.log2(v)) if v>0 else 0.0
    fig,ax=plt.subplots(figsize=(max(9,0.34*len(tags)),3.6))
    im=ax.imshow(L,aspect='auto',cmap='Greys',vmin=0,vmax=np.ceil(L.max()))
    ax.set_xticks(range(len(tags))); ax.set_yticks(range(len(cl_order)))
    ax.set_xticklabels(tags,rotation=90,fontsize=8)
    ax.set_yticklabels([f'C{i+1}  '+SHORT.get(meta[c]['name'],'') for i,c in enumerate(cl_order)],fontsize=9)
    for j,t in enumerate(tags): ax.get_xticklabels()[j].set_color(meta[owner[t]]['color'])
    for i,c in enumerate(cl_order): ax.get_yticklabels()[i].set_color(meta[c]['color']); ax.get_yticklabels()[i].set_fontweight('bold')
    ax.set_xticks(np.arange(-.5,len(tags),1),minor=True); ax.set_yticks(np.arange(-.5,len(cl_order),1),minor=True)
    ax.grid(which='minor',color='white',lw=1.2); ax.tick_params(which='minor',length=0)
    for i,c in enumerate(cl_order):
        for j,t in enumerate(tags):
            q=fdr.loc[c,t] if t in fdr.columns else 1; v=oe.loc[c,t] if t in oe.columns else 0
            if q<0.05 and v>1: ax.plot(j,i,'o',mfc='white',mec='#333',mew=0.6,ms=4.2,zorder=5)
    cb=fig.colorbar(im,ax=ax,fraction=0.025,pad=0.01); cb.set_label('log$_2$ fold-enrichment',fontsize=9,color='#8a8a86'); cb.ax.tick_params(labelsize=8)
    ax.set_title(title,fontsize=12,fontweight='bold',pad=10,color='#6b6b67')
    save(fig,f'enrichment_{name}')

### PCoA — principal coordinate analysis of the co-tuning space
Classical (metric) MDS: embeds the molecules from their pairwise **Jaccard** dissimilarity (shared receptor activation — the same distance used for clustering) into Euclidean axes that best preserve those distances. PCoA rather than PCA because Jaccard is non-Euclidean. Axes are ordered by variance explained (eigenvalue share); points close together activate similar receptor sets; colour = PAM cluster.

In [10]:
def pcoa(D):
    D=np.asarray(D,float); n=len(D); D2=D**2
    J=np.eye(n)-np.ones((n,n))/n; B=-0.5*J.dot(D2).dot(J)
    ev,evec=np.linalg.eigh((B+B.T)/2); idx=np.argsort(ev)[::-1]; ev=ev[idx]; evec=evec[:,idx]
    pos=ev>1e-9; return evec[:,pos]*np.sqrt(ev[pos]), ev
def pcoa_plot(Xn,lab,meta,cl_order,name,title):
    Dj=np.nan_to_num(squareform(pdist(Xn.T,"jaccard"))); coords,ev=pcoa(Dj)
    tot=ev[ev>0].sum(); v1,v2=100*ev[0]/tot,100*ev[1]/tot
    neg=100*(-ev[ev<0].sum())/(np.abs(ev).sum())
    fig,ax=plt.subplots(figsize=(7.6,6.8))
    for c in cl_order:
        m=lab==c; P=coords[m,:2]
        ax.scatter(P[:,0],P[:,1],s=15,color=meta[c]["color"],alpha=0.75,edgecolor="none",label=f'{meta[c]["name"]} (n={meta[c]["size"]})',zorder=3)
    ax.axhline(0,color="#e5e5e2",lw=0.6,zorder=0); ax.axvline(0,color="#e5e5e2",lw=0.6,zorder=0)
    ax.set_xlabel(f"PCo1 ({v1:.1f}%)",fontsize=10.5,color="#8a8a86"); ax.set_ylabel(f"PCo2 ({v2:.1f}%)",fontsize=10.5,color="#8a8a86")
    for sp in ["top","right"]: ax.spines[sp].set_visible(False)
    ax.tick_params(labelsize=8); ax.legend(frameon=False,fontsize=8.3,loc="best",labelcolor="#6b6b67")
    ax.set_title(title,fontsize=12.5,fontweight="bold",color="#6b6b67")
    ax.text(0.99,0.01,f"non-Euclidean (neg. eigenvalue mass {neg:.0f}%)",transform=ax.transAxes,ha="right",va="bottom",fontsize=7,color="#b5b5b0")
    save(fig,f"pcoa_{name}")

## 6 · Build the four matrices

### Enrichment bar plot — fold enrichment, q-value labels (screenshot style)

In [11]:
def _fmtq(q):
    if q<1e-5: return 'q<1e-5'
    if q<1e-3: return 'q=%.0e'%q
    return 'q=%.3f'%q

def enrichment_bars(enr,sig,meta,cl_order,name,title):
    rows=[]
    for c in cl_order:
        for r in sig[sig.cluster==c].sort_values('obs_exp',ascending=False).itertuples():
            rows.append((c,r.tag,r.obs_exp,r.fdr))
    if not rows: return
    ys=[]; labels=[]; colors=[]; folds=[]; qs=[]; y=0; last=None
    for c,tag,fold,q in rows:
        if last is not None and c!=last: y-=1
        ys.append(y); labels.append(tag); colors.append(meta[c]['color']); folds.append(fold); qs.append(q); y-=1; last=c
    ys=np.array(ys); folds=np.array(folds)
    fig,ax=plt.subplots(figsize=(8.6,0.3*(len(labels)+len(cl_order))+1))
    ax.barh(ys,folds,color=colors,alpha=0.92,height=0.74)
    ax.axvline(1,color='#8a8a86',ls='--',lw=1.0)
    xm=folds.max()
    for yi,f,q in zip(ys,folds,qs): ax.text(f+xm*0.015,yi,_fmtq(q),va='center',ha='left',fontsize=7.6,color='#8a8a86')
    ax.set_yticks(ys); ax.set_yticklabels(labels,fontsize=8.5)
    for tl,c in zip(ax.get_yticklabels(),colors): tl.set_color(c)
    ax.set_xlim(0,xm*1.30); ax.set_ylim(ys.min()-1,ys.max()+1)
    ax.set_xlabel('fold enrichment of odour descriptor  (observed / expected)',fontsize=10)
    ax.set_title(title,fontsize=12,fontweight='bold',color='#6b6b67')
    for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
    ax.tick_params(left=False)
    save(fig,f'enrichbar_{name}')

### Two-panel chemistry figure — Class I vs II (receptor-level) + clusters (Kruskal-Wallis)

In [12]:
from scipy.stats import kruskal
def _cliffs(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.sign(a[:,None]-b[None,:]).mean())
def chem_panels(lig, Xn, lab, meta, cl_order, name, min_lig=3):
    keys=['OxygenAtoms','HBD','TPSA','RotBonds','AromaticRings','LogP','RingCount','FracCSP3']
    vals={k:np.full(len(lig),np.nan) for k in keys}
    for i,sid in enumerate(lig):
        sm=sml2smiles.get(sid); mol=Chem.MolFromSmiles(sm) if isinstance(sm,str) else None
        if mol is None: continue
        vals['OxygenAtoms'][i]=sum(a.GetAtomicNum()==8 for a in mol.GetAtoms())
        vals['HBD'][i]=Lipinski.NumHDonors(mol); vals['TPSA'][i]=rdMolDescriptors.CalcTPSA(mol)
        vals['RotBonds'][i]=Descriptors.NumRotatableBonds(mol); vals['AromaticRings'][i]=rdMolDescriptors.CalcNumAromaticRings(mol)
        vals['LogP'][i]=Crippen.MolLogP(mol); vals['RingCount'][i]=rdMolDescriptors.CalcNumRings(mol); vals['FracCSP3'][i]=rdMolDescriptors.CalcFractionCSP3(mol)
    valid=~np.isnan(vals['LogP'])
    # Panel A: receptor-level (mean chemistry of each receptor's ligand set), Class I vs II, z-scored
    descA=[('OxygenAtoms','oxygen\natoms'),('HBD','H-bond\ndonors'),('TPSA','polar surf.\n(TPSA)'),
           ('RotBonds','rotatable\nbonds'),('AromaticRings','aromatic\nrings'),('LogP','lipophilicity\n(LogP)')]
    reccls=[]; recval={d[0]:[] for d in descA}
    for r in range(len(hum_pids)):
        b=np.where(Xn[r]>0)[0]; b=b[valid[b]]
        if len(b)<min_lig: continue   # stable per-receptor mean
        reccls.append(pid2class[hum_pids[r]])
        for k in recval: recval[k].append(float(np.nanmean(vals[k][b])))
    reccls=np.array(reccls)
    z={k:(np.array(v)-np.mean(v))/(np.std(v)+1e-9) for k,v in recval.items()}
    nI=int((reccls=='I').sum()); nII=int((reccls=='II').sum())
    # Panel B: cluster-level (odorants grouped by co-tuning cluster), Kruskal-Wallis
    descB=[('RingCount','rings'),('FracCSP3','sp\u00b3 fraction'),('LogP','lipophilicity (LogP)')]
    kwset=['RingCount','FracCSP3','LogP','TPSA','AromaticRings','OxygenAtoms','RotBonds','HBD']
    kwp={}
    for k in kwset:
        try: kwp[k]=kruskal(*[vals[k][(lab==c)&valid] for c in cl_order])[1]
        except Exception: kwp[k]=np.nan
    ks=list(kwp); pp=np.array([kwp[k] for k in ks]); mm=len(pp); oo=np.argsort(pp); q=np.empty(mm); rk=pp[oo]
    q[oo]=np.clip(np.minimum.accumulate((rk*mm/(np.arange(mm)+1))[::-1])[::-1],0,1); qmap=dict(zip(ks,q))
    fq=lambda x: ('q=%.0e'%x if x<1e-3 else 'q=%.2f'%x)
    fig=plt.figure(figsize=(17,5.4))
    gs=fig.add_gridspec(1,5,width_ratios=[2.7,0.28,1,1,1],wspace=0.42,left=0.05,right=0.995,top=0.80,bottom=0.15)
    axA=fig.add_subplot(gs[0,0])
    for i,(k,lb) in enumerate(descA):
        zI=z[k][reccls=='I']; zII=z[k][reccls=='II']
        vp=axA.violinplot([zI,zII],positions=[i*2,i*2+0.9],widths=0.8,showmedians=False,showextrema=False)
        for pc,cc in zip(vp['bodies'],[CLASS_I_COL,CLASS_II_COL]): pc.set_facecolor(cc); pc.set_alpha(0.78); pc.set_edgecolor('none')
        for pos,vv in [(i*2,zI),(i*2+0.9,zII)]:
            a1,md,a3=np.percentile(vv,[25,50,75]); axA.vlines(pos,a1,a3,color='#333',lw=3); axA.plot(pos,md,'o',mfc='white',mec='#333',ms=4,zorder=5)
        d=_cliffs(zI,zII); pv=mannwhitneyu(zI,zII).pvalue
        s='***' if pv<1e-3 else '**' if pv<1e-2 else '*' if pv<0.05 else 'ns'
        axA.text(i*2+0.45,3.6,s,ha='center',fontsize=10,color='#6b6b67')
        axA.text(i*2+0.45,3.05,'\u03b4=%+.2f'%d,ha='center',fontsize=8,color='#8a8a86')
    axA.axhline(0,color='#ccc',lw=0.8,zorder=0)
    axA.set_xticks([i*2+0.45 for i in range(len(descA))]); axA.set_xticklabels([lb for _,lb in descA],fontsize=9)
    axA.set_ylabel("chemistry of the receptor's ligand set (z-score)",fontsize=10); axA.set_ylim(-3,4.2)
    for sp in ['top','right']: axA.spines[sp].set_visible(False)
    axA.legend(handles=[Patch(fc=CLASS_I_COL,label='Class I  (n=%d)'%nI),Patch(fc=CLASS_II_COL,label='Class II (n=%d)'%nII)],frameon=False,fontsize=9,loc='lower left')
    axA.set_title('A   The two ancient receptor classes read different chemistry\nClass I: polar, H-bonding aliphatics \u00b7 Class II: aromatic, lipophilic',fontsize=11,fontweight='bold',loc='left',color='#6b6b67')
    axB=[fig.add_subplot(gs[0,2+j]) for j in range(3)]
    for ax,(k,lb) in zip(axB,descB):
        data=[vals[k][(lab==c)&valid] for c in cl_order]
        vp=ax.violinplot(data,positions=range(len(cl_order)),widths=0.85,showmedians=False,showextrema=False)
        for pc,c in zip(vp['bodies'],cl_order): pc.set_facecolor(meta[c]['color']); pc.set_alpha(0.78); pc.set_edgecolor('none')
        for j,vv in enumerate(data):
            a1,md,a3=np.percentile(vv,[25,50,75]); ax.vlines(j,a1,a3,color='#333',lw=2.5); ax.plot(j,md,'o',mfc='white',mec='#333',ms=4,zorder=5)
        H=kruskal(*data)[0]
        ax.set_title('%s\nH=%.0f, %s'%(lb,H,fq(qmap.get(k,np.nan))),fontsize=10,color='#6b6b67')
        ax.set_xticks(range(len(cl_order))); ax.set_xticklabels(['C%d'%(i+1) for i in range(len(cl_order))],fontsize=8.5)
        for tl,c in zip(ax.get_xticklabels(),cl_order): tl.set_color(meta[c]['color'])
        for sp in ['top','right']: ax.spines[sp].set_visible(False)
    axB[0].set_ylabel('descriptor value',fontsize=10)
    fig.text(0.625,0.9,'B   The activation clusters are chemically distinct',fontsize=11,fontweight='bold',color='#6b6b67')
    fig.text(0.625,0.85,'similar activation \u21d2 similar chemistry',fontsize=9,color='#a5a5a0')
    save(fig,f'chemistry_{name}')

In [13]:
ridx={p:i for i,p in enumerate(hum_pids)}; cidx={c:j for j,c in enumerate(lig_cols)}
base=(Pv>=0.50).astype(int); calib=(Pv>CALIB_THR).astype(int)
hybrid_05=base.copy(); hybrid_083=calib.copy()
for r in exp.itertuples():
    hybrid_05[ridx[r.pid],cidx[r.smiles_id]]=int(r.Responsive)
    hybrid_083[ridx[r.pid],cidx[r.smiles_id]]=int(r.Responsive)
print('densities  base=%.4f calib=%.4f hybrid_0.5=%.4f hybrid_0.83=%.4f'%(base.mean(),calib.mean(),hybrid_05.mean(),hybrid_083.mean()))

def run_matrix(name,X,title,subtitle):
    lig,Xn,lab,enr,sig,meta,cl_order=cluster_and_enrich(X)
    enr.sort_values(['cluster','fdr']).to_csv(f'{OUTDIR}/enrichment_{name}.csv',index=False)
    activation_heatmap(lig,Xn,lab,meta,cl_order,name,title,subtitle,'cotuning')
    activation_heatmap(lig,Xn,lab,meta,cl_order,name,title,subtitle,'tree')
    spider(enr,sig,meta,cl_order,name,'Perceptual enrichment — '+name)
    enrichment_heatmap(enr,sig,meta,cl_order,name,'Perceptual enrichment — '+name)
    enrichment_bars(enr,sig,meta,cl_order,name,'Odour-descriptor enrichment — '+name)
    chem_panels(lig,Xn,lab,meta,cl_order,name)
    pcoa_plot(Xn,lab,meta,cl_order,name,'PCoA of odorant co-tuning (Jaccard) — '+name)
    print('%-12s sizes %s  #sig %d'%(name,sorted([meta[c]['size'] for c in cl_order],reverse=True),len(sig)))

densities  base=0.1356 calib=0.0512 hybrid_0.5=0.1266 hybrid_0.83=0.0481


## 7 · Baseline / Calibrated / Hybrid 0.5 / Hybrid 0.83

In [14]:
run_matrix('baseline',   base,       'Baseline landscape (p ≥ 0.50)',                       'released prediction file')
run_matrix('calibrated', calib,      'Calibrated landscape (p > 0.83)',                      'high-confidence predictions')
run_matrix('hybrid_0.5', hybrid_05,  'Hybrid landscape (M2OR truth + baseline p ≥ 0.50 fill)','experimental where available, else p ≥ 0.50')
run_matrix('hybrid_0.83',hybrid_083, 'Hybrid landscape (M2OR truth + calibrated p > 0.83 fill)','experimental where available, else p > 0.83')

/tmp/ipykernel_2663979/968434317.py:13: UserWarning: Glyph 8658 (\N{RIGHTWARDS DOUBLE ARROW}) missing from font(s) Liberation Sans.
  fig.savefig(f'{OUTDIR}/{name}.svg', bbox_inches='tight'); plt.close(fig)


baseline     sizes [184, 156, 132, 130, 84, 65]  #sig 37


/tmp/ipykernel_2663979/968434317.py:13: UserWarning: Glyph 8658 (\N{RIGHTWARDS DOUBLE ARROW}) missing from font(s) Liberation Sans.
  fig.savefig(f'{OUTDIR}/{name}.svg', bbox_inches='tight'); plt.close(fig)


calibrated   sizes [141, 135, 133, 125, 109, 83]  #sig 40


/tmp/ipykernel_2663979/968434317.py:13: UserWarning: Glyph 8658 (\N{RIGHTWARDS DOUBLE ARROW}) missing from font(s) Liberation Sans.
  fig.savefig(f'{OUTDIR}/{name}.svg', bbox_inches='tight'); plt.close(fig)


hybrid_0.5   sizes [167, 150, 141, 123, 114, 57]  #sig 29


/tmp/ipykernel_2663979/968434317.py:13: UserWarning: Glyph 8658 (\N{RIGHTWARDS DOUBLE ARROW}) missing from font(s) Liberation Sans.
  fig.savefig(f'{OUTDIR}/{name}.svg', bbox_inches='tight'); plt.close(fig)


hybrid_0.83  sizes [145, 144, 141, 119, 103, 79]  #sig 43


## 8 · Chemical space of Class I- vs Class II-preferring odorants (one figure per matrix)

In [15]:
_FG={'carboxylic acid':'[CX3](=O)[OX2H1]','ester':'[CX3](=O)[OX2H0][#6]','aldehyde':'[CX3H1](=O)[#6]',
     'ketone':'[#6][CX3](=O)[#6]','alcohol':'[#6;!$([CX3]=O)][OX2H1]','ether':'[OD2]([#6])[#6]',
     'aromatic ring':'c1ccccc1','amine':'[NX3;!$([NX3][CX3]=[OX1])]','sulfur':'[#16]'}
_FGP={k:Chem.MolFromSmarts(v) for k,v in _FG.items()}

def _cliffs(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.sign(a[:,None]-b[None,:]).mean())   # >0 => Class I larger

def chemical_space(Xbind,name):
    isI=np.array([pid2class[p]=='I' for p in hum_pids]); isII=~isI; NI,NII=isI.sum(),isII.sum()
    recs=[]
    for j,sid in enumerate(lig_cols):
        col=Xbind[:,j]; nI=int(col[isI].sum()); nII=int(col[isII].sum())
        if nI==0 and nII==0: continue
        pref='I' if (nI/NI)>(nII/NII) else 'II'
        sm=sml2smiles.get(sid); mol=Chem.MolFromSmiles(sm) if isinstance(sm,str) else None
        if mol is None: continue
        d=dict(sml=sid,pref=pref,MolWt=Descriptors.MolWt(mol),LogP=Crippen.MolLogP(mol),TPSA=rdMolDescriptors.CalcTPSA(mol),
            HBD=Lipinski.NumHDonors(mol),HBA=Lipinski.NumHAcceptors(mol),AromaticRings=rdMolDescriptors.CalcNumAromaticRings(mol),
            RotBonds=Descriptors.NumRotatableBonds(mol),RingCount=rdMolDescriptors.CalcNumRings(mol),
            FracCSP3=rdMolDescriptors.CalcFractionCSP3(mol),Heteroatoms=Lipinski.NumHeteroatoms(mol))
        for g,patt in _FGP.items(): d[g]=int(mol.HasSubstructMatch(patt))
        recs.append(d)
    df=pd.DataFrame(recs); df.to_csv(f'{OUTDIR}/chemspace_{name}_descriptors.csv',index=False)
    gI=df[df.pref=='I']; gII=df[df.pref=='II']
    props=[('MolWt','MW (Da)'),('LogP','cLogP'),('TPSA','TPSA (Å²)'),('HBD','H-bond donors'),('HBA','H-bond acceptors'),
           ('AromaticRings','aromatic rings'),('RotBonds','rotatable bonds'),('RingCount','ring count'),('FracCSP3','fraction sp³'),('Heteroatoms','heteroatoms')]

    # (a) descriptor violins
    fig,axes=plt.subplots(2,5,figsize=(16,7)); axes=axes.ravel()
    for ax,(k,lab) in zip(axes,props):
        a,b=gI[k].values,gII[k].values; parts=ax.violinplot([a,b],showmedians=True,widths=0.85)
        for pc,col in zip(parts['bodies'],[CLASS_I_COL,CLASS_II_COL]): pc.set_facecolor(col); pc.set_alpha(0.65); pc.set_edgecolor('#333')
        for key in ['cmedians','cmaxes','cmins','cbars']: parts[key].set_color('#555'); parts[key].set_linewidth(1.0)
        try: p=mannwhitneyu(a,b,alternative='two-sided').pvalue
        except Exception: p=np.nan
        ax.set_title('%s\nMWU p=%.1e'%(lab,p),fontsize=9.5,color='#6b6b67')
        ax.set_xticks([1,2]); ax.set_xticklabels(['Class I','Class II'],fontsize=8.5); ax.tick_params(axis='y',labelsize=8)
        for sp in ['top','right']: ax.spines[sp].set_visible(False)
    fig.suptitle('Chemical space of Class I- vs Class II-preferring odorants — %s  ·  n(I)=%d, n(II)=%d'%(name,len(gI),len(gII)),fontsize=13,fontweight='bold',y=1.0,color='#6b6b67')
    fig.tight_layout(rect=[0,0,1,0.97]); save(fig,f'chemspace_{name}')

    # (b) functional groups (incl. carboxylic acids) — all in one plot
    from scipy.stats import fisher_exact
    groups=list(_FG.keys())
    fracI=[100*gI[g].mean() for g in groups]; fracII=[100*gII[g].mean() for g in groups]
    pg=[]
    for g in groups:
        a1=int(gI[g].sum()); a2=int(gII[g].sum())
        try: pg.append(fisher_exact([[a1,len(gI)-a1],[a2,len(gII)-a2]])[1])
        except Exception: pg.append(np.nan)
    o=np.argsort(fracI); groups=[groups[i] for i in o]; fracI=[fracI[i] for i in o]; fracII=[fracII[i] for i in o]; pg=[pg[i] for i in o]
    yy=np.arange(len(groups)); hh=0.38
    fig,ax=plt.subplots(figsize=(8.6,0.6*len(groups)+1.2))
    ax.barh(yy+hh/2,fracI,height=hh,color=CLASS_I_COL,alpha=0.9,label='Class I-preferring')
    ax.barh(yy-hh/2,fracII,height=hh,color=CLASS_II_COL,alpha=0.9,label='Class II-preferring')
    xm=max(max(fracI),max(fracII))
    for yi,p in zip(yy,pg):
        s='***' if p<1e-3 else '**' if p<1e-2 else '*' if p<0.05 else 'ns'
        ax.text(xm*1.02,yi,s,va='center',fontsize=9,color='#8a8a86')
    ax.set_yticks(yy); ax.set_yticklabels(groups,fontsize=9)
    ax.set_xlabel('% of odorants containing group',fontsize=10); ax.set_xlim(0,xm*1.15)
    ax.legend(frameon=False,fontsize=9,loc='lower right')
    ax.set_title('Functional groups — %s'%name,fontsize=12,fontweight='bold',color='#6b6b67')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    save(fig,f'chemspace_{name}_groups')

    # (c) effect sizes — what differs most (lipophilicity, rings, sp3, ...)
    desc=[k for k,_ in props]; labm=dict(props); eff=[]; pv=[]
    for k in desc:
        eff.append(_cliffs(gI[k].values,gII[k].values))
        try: pv.append(mannwhitneyu(gI[k].values,gII[k].values).pvalue)
        except Exception: pv.append(np.nan)
    pv=np.array(pv); mm=len(pv); oo=np.argsort(pv); q=np.empty(mm); rk=pv[oo]
    q[oo]=np.clip(np.minimum.accumulate((rk*mm/(np.arange(mm)+1))[::-1])[::-1],0,1)
    idx=np.argsort(eff); eff=np.array(eff)[idx]; dl=[labm[desc[i]] for i in idx]; q=q[idx]
    fig,ax=plt.subplots(figsize=(8,0.5*len(dl)+1.2))
    cols=[CLASS_I_COL if e>0 else CLASS_II_COL for e in eff]
    ax.barh(range(len(dl)),eff,color=cols,alpha=0.9); ax.axvline(0,color='#c9c9c4',lw=0.8)
    xr=max(abs(eff.min()),abs(eff.max()),0.05)
    for i,(e,qq) in enumerate(zip(eff,q)):
        s='***' if qq<1e-3 else '**' if qq<1e-2 else '*' if qq<0.05 else 'ns'
        ax.text(e+0.03*xr*(1 if e>=0 else -1),i,s,va='center',ha='left' if e>=0 else 'right',fontsize=8.5,color='#8a8a86')
    ax.set_yticks(range(len(dl))); ax.set_yticklabels(dl,fontsize=9); ax.set_xlim(-xr*1.3,xr*1.3)
    ax.set_xlabel('effect size (Cliff delta)    <- Class II higher   |   Class I higher ->',fontsize=9.5)
    ax.set_title('What differs most — %s'%name,fontsize=12,fontweight='bold',color='#6b6b67')
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
    save(fig,f'chemspace_{name}_effects')
    print('chemspace %-11s n(I)=%d n(II)=%d'%(name,len(gI),len(gII)))

for _nm,_X in [('baseline',base),('calibrated',calib),('hybrid_0.5',hybrid_05),('hybrid_0.83',hybrid_083)]:
    chemical_space(_X,_nm)

chemspace baseline    n(I)=443 n(II)=308
chemspace calibrated  n(I)=337 n(II)=389
chemspace hybrid_0.5  n(I)=449 n(II)=303
chemspace hybrid_0.83 n(I)=341 n(II)=390


## Notes
- `hybrid_0.5` and `hybrid_0.83` share the same 23,782 experimental cells; they differ only in the predicted fill (p ≥ 0.50 vs p > 0.83).
- Each matrix yields `heatmap_{name}` (co-tuning row order) and `heatmap_{name}_tree_order` (phylogenetic row order); enrichment shown both as `spider_{name}` and `enrichment_{name}`.
- Chemical space uses the calibrated matrix to assign preferential class binding; edit `chemical_space(...)` to use another matrix or exclusive binders.